# Store supporting features

# Set up

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import sys

import pandas as pd
import redis
from dotenv import load_dotenv
from pydantic import BaseModel
from tqdm.auto import tqdm

sys.path.insert(0, "..")

from src.cfg import ConfigLoader
from src.id_mapper import IDMapper
from src.io_utils import init_s3_client

load_dotenv()

True

# Controller

In [3]:
cfg = ConfigLoader("../cfg/common.yaml")
cfg

{
  "run": {
    "author": "",
    "testing": false,
    "log_to_mlflow": true,
    "experiment_name": null,
    "run_name": null,
    "run_persist_dir": null,
    "random_seed": 41
  },
  "data": {
    "hf_datasets": {
      "name": "McAuley-Lab/Amazon-Reviews-2023",
      "mcauley_variant": "Books"
    },
    "train_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/train.parquet",
    "val_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/val.parquet",
    "idm_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/idm.json",
    "metadata_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/metadata.parquet",
    "train_features_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/train_features.parquet",
    "val_features_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/val_features.parquet",
    "full_features_neg_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/full_features_neg_sampling_df.parquet",
    "train_features_neg_fp": "/home/dvq/

# Load input data

In [4]:
if not os.path.exists(cfg.data.train_features_fp):
    s3 = init_s3_client()
    bucket_name = cfg.data.bucket_name
    train_key = cfg.data.train_features_fp.split("/")[-1]
    val_key = cfg.data.val_features_fp.split("/")[-1]
    idm_key = cfg.data.idm_fp.split("/")[-1]

    s3.download_file(bucket_name, train_key, cfg.data.train_features_fp)
    s3.download_file(bucket_name, val_key, cfg.data.val_features_fp)
    s3.download_file(bucket_name, idm_key, cfg.data.idm_fp)

In [5]:
train_features_df = pd.read_parquet(cfg.data.train_features_fp)
val_features_df = pd.read_parquet(cfg.data.val_features_fp)
idm = IDMapper().load(cfg.data.idm_fp)
full_df = pd.concat([train_features_df, val_features_df], axis=0)
full_df

,user_id,parent_asin,rating,timestamp,user_indice,item_indice,item_sequence
0,AE224PFXAEAT66IXX43GRJSWHXCA,0399159312,2.0,1373291889000,0,1251,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1...."
1,AE224PFXAEAT66IXX43GRJSWHXCA,B000FA5TTW,1.0,1382077065000,0,3363,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1...."
2,AE224PFXAEAT66IXX43GRJSWHXCA,030758836X,1.0,1424138603000,0,499,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1...."
3,AE224PFXAEAT66IXX43GRJSWHXCA,B00MSRW6SM,4.0,1437924147000,0,5410,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 125..."
4,AE224PFXAEAT66IXX43GRJSWHXCA,B00A18VD7A,1.0,1464603674000,0,4639,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1251.0, 3..."
...,...,...,...,...,...,...,...
3565,AHZLM4RDKSICEFEAYEQQRZW45BPA,B08CV9SPDQ,5.0,1657516186943,19672,7335,"[-1, -1, -1, -1, 6121, 5345, 5735, 6479, 6758,..."
3566,AHZLQPSPG675BABC5R5NJW6KG3WQ,B07D6PZ6P1,5.0,1635813519329,19673,6837,"[-1, -1, -1, -1, -1, 4387, 4340, 4578, 6829, 7..."
3567,AHZNQ34GWKKLJN53IDXLAX22OBJQ,B07ZJ2VHBB,5.0,1652978096579,19682,7208,"[-1, -1, -1, -1, -1, 4195, 7126, 7175, 6817, 6..."
3568,AHZNQ34GWKKLJN53IDXLAX22OBJQ,B00PG8UCGS,5.0,1654707732874,19682,5521,"[-1, -1, -1, -1, 4195, 7126, 7175, 6817, 6807,..."


In [6]:
latest_df = full_df.assign(
    recency=lambda df: df.groupby(cfg.data.user_col)[cfg.data.timestamp_col].rank(
        method="first", ascending=False
    )
).loc[lambda df: df["recency"].eq(1)]
latest_df

,user_id,parent_asin,rating,timestamp,user_indice,item_indice,item_sequence,recency
5,AE224PFXAEAT66IXX43GRJSWHXCA,B007Q4750Q,5.0,1528110039684,0,4428,"[-1.0, -1.0, -1.0, -1.0, -1.0, 1251.0, 3363.0,...",1.0
16,AE225Y3KDZ44DHLUKLE4RJ63HC5Q,B07FDDPFPN,5.0,1551299380152,1,6870,"[5841.0, 6248.0, 6295.0, 6471.0, 5169.0, 6465....",1.0
22,AE226YVDC3MAGJZMZ4IBGE7RFJSQ,B08C6Z2C6G,5.0,1623117469062,2,7330,"[-1.0, -1.0, -1.0, -1.0, -1.0, 4557.0, 5114.0,...",1.0
28,AE22EJZ4354VB7MN4IE2CDGHA2DQ,1250133572,4.0,1628101019843,3,2522,"[-1.0, -1.0, -1.0, -1.0, -1.0, 3324.0, 2980.0,...",1.0
40,AE22O3TURLPFCJKL7YCX5CPF22OA,1501190075,3.0,1546520172484,4,3055,"[156.0, 499.0, 4139.0, 4749.0, 4656.0, 5342.0,...",1.0
...,...,...,...,...,...,...,...,...
3564,AHZL7PB4UL47R3E24ENBLOLK6DVA,B006JTTK3O,5.0,1656656442094,19670,4337,"[3946, 6552, 375, 379, 5867, 4226, 4804, 4610,...",1.0
3565,AHZLM4RDKSICEFEAYEQQRZW45BPA,B08CV9SPDQ,5.0,1657516186943,19672,7335,"[-1, -1, -1, -1, 6121, 5345, 5735, 6479, 6758,...",1.0
3566,AHZLQPSPG675BABC5R5NJW6KG3WQ,B07D6PZ6P1,5.0,1635813519329,19673,6837,"[-1, -1, -1, -1, -1, 4387, 4340, 4578, 6829, 7...",1.0
3568,AHZNQ34GWKKLJN53IDXLAX22OBJQ,B00PG8UCGS,5.0,1654707732874,19682,5521,"[-1, -1, -1, -1, 4195, 7126, 7175, 6817, 6807,...",1.0


# Load recent interacted items into Redis

In [7]:
r = redis.Redis(host=cfg.redis.host, port=cfg.redis.port, db=0, decode_responses=True)
assert (
    r.ping()
), f"Redis at {cfg.redis.host}:{cfg.redis.port} is not running, please make sure you have started the Redis docker service"

In [8]:
latest_df[[cfg.data.user_col, cfg.data.item_col, "item_sequence"]]

,user_id,parent_asin,item_sequence
5,AE224PFXAEAT66IXX43GRJSWHXCA,B007Q4750Q,"[-1.0, -1.0, -1.0, -1.0, -1.0, 1251.0, 3363.0,..."
16,AE225Y3KDZ44DHLUKLE4RJ63HC5Q,B07FDDPFPN,"[5841.0, 6248.0, 6295.0, 6471.0, 5169.0, 6465...."
22,AE226YVDC3MAGJZMZ4IBGE7RFJSQ,B08C6Z2C6G,"[-1.0, -1.0, -1.0, -1.0, -1.0, 4557.0, 5114.0,..."
28,AE22EJZ4354VB7MN4IE2CDGHA2DQ,1250133572,"[-1.0, -1.0, -1.0, -1.0, -1.0, 3324.0, 2980.0,..."
40,AE22O3TURLPFCJKL7YCX5CPF22OA,1501190075,"[156.0, 499.0, 4139.0, 4749.0, 4656.0, 5342.0,..."
...,...,...,...
3564,AHZL7PB4UL47R3E24ENBLOLK6DVA,B006JTTK3O,"[3946, 6552, 375, 379, 5867, 4226, 4804, 4610,..."
3565,AHZLM4RDKSICEFEAYEQQRZW45BPA,B08CV9SPDQ,"[-1, -1, -1, -1, 6121, 5345, 5735, 6479, 6758,..."
3566,AHZLQPSPG675BABC5R5NJW6KG3WQ,B07D6PZ6P1,"[-1, -1, -1, -1, -1, 4387, 4340, 4578, 6829, 7..."
3568,AHZNQ34GWKKLJN53IDXLAX22OBJQ,B00PG8UCGS,"[-1, -1, -1, -1, 4195, 7126, 7175, 6817, 6807,..."


In [9]:
for i, row in tqdm(latest_df.iterrows(), total=latest_df.shape[0]):
    prev_item_indices = [int(item) for item in row["item_sequence"] if item != -1]
    prev_item_ids = [idm.get_item_id(idx) for idx in prev_item_indices]
    updated_item_sequences = prev_item_ids + [row[cfg.data.item_col]]
    user_id = row[cfg.data.user_col]
    key = cfg.redis.keys.recent_key_prefix + user_id
    value = "__".join(updated_item_sequences)
    r.set(key, value)

  0%|          | 0/19734 [00:00<?, ?it/s]

In [10]:
test_user_id = latest_df.sample(1)[cfg.data.user_col].values[0]
r.get(cfg.redis.keys.recent_key_prefix + test_user_id)

'B078GWN38X__B078JJFFGK__B07ZDG34ZC__B079QG6L98__B00M9GZTXG__B07CWSSFL3__B0031W1E86__B07LF2YL9S__B07BJZJ34M__B077XVF99N__B07F668MBT'

# Load popular items into Redis

In [11]:
popular_recs = (
    full_df.groupby(cfg.data.item_col).size().sort_values(ascending=False).head(cfg.eval.top_k_retrieve)
)
popular_recs

parent_asin
B00L9B7IKE    1248
B00JO8PEN2     857
B006LSZECO     774
B00DPM7TIG     646
B00CNQ7HAU     598
              ... 
B004TI5N38     152
B00IB5BSBG     151
B087PKF9LZ     151
1476746583     150
0375842209     149
Length: 100, dtype: int64

In [12]:
key = cfg.redis.keys.popular_key
value = json.dumps(
    {
        "rec_item_ids": popular_recs.index.tolist(),
        "rec_scores": popular_recs.values.tolist(),
    }
)
r.set(key, value)

True

## Test get data from Redis

In [13]:
redis_data = json.loads(r.get(key))
print(redis_data)
assert len(redis_data["rec_item_ids"]) == cfg.eval.top_k_retrieve

{'rec_item_ids': ['B00L9B7IKE', 'B00JO8PEN2', 'B006LSZECO', 'B00DPM7TIG', 'B00CNQ7HAU', 'B016ZNRC0Q', 'B00YTXTIDO', 'B019G6DSDE', 'B06Y1264PX', 'B016JC0THQ', 'B007SGLZP8', 'B00UEKRTW8', 'B01KXQ8SS6', 'B01C1LUFFK', 'B01COJUEZ0', 'B01B1OGQH4', 'B01M7XPGYE', 'B00AA20E5Y', 'B00C2WDD5I', 'B01BU1ITMI', 'B00K7MCE3C', 'B072BLVM83', 'B01L1CEZ6K', 'B07415PPP1', 'B00GEEB52S', 'B0050DIWFC', 'B074QL7WNM', 'B01MYDJB6R', 'B00A6JLDJ2', 'B00IJJUIMY', 'B00NMPN46W', 'B0146LBFIE', 'B00O2BKKUS', 'B06Y55Z36S', 'B07416NFHL', 'B00INIQTY2', 'B01EN506CO', 'B0027MJU00', 'B07DHMNY7H', 'B00SLWQGUM', 'B00DPM90C4', 'B00ZETXO0K', 'B00BAXFACO', 'B011G3HI9U', 'B07D2C6J4K', 'B00UCLPCE6', 'B00U6DNY5O', 'B00DMCPQUW', 'B078JG8M8P', 'B00H58VGIA', 'B00GU2RLMC', 'B00HNFD9VC', 'B00P42WROG', 'B0141ZP33S', 'B00PG8UCGS', 'B0076DELIG', 'B00IJJUIOM', 'B00S5K0CAU', 'B000OZ0NXA', 'B017RBIZGK', 'B007FEFLTO', '0735219095', 'B00IRIR7K8', 'B006VFLIYK', '1594633665', 'B01MUDRSND', 'B01COJUGOE', 'B00FJ3AC10', '030758836X', 'B07R3QYGHY', 'B